In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 06 · Productos Gold — Trust & Safety
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Publicar productos analíticos certificados para consumo
# MAGIC de Power BI a partir de:
# MAGIC
# MAGIC - Silver conformada
# MAGIC - Evaluaciones Trust & Safety
# MAGIC - Risk Consolidation
# MAGIC - Catálogo y versiones de reglas
# MAGIC - Evaluation Runs
# MAGIC
# MAGIC ## Importante
# MAGIC
# MAGIC Gold NO calcula nuevamente:
# MAGIC
# MAGIC - signal_count
# MAGIC - weighted_risk_score
# MAGIC - signal_level
# MAGIC - requires_investigation
# MAGIC - investigation_priority
# MAGIC - risk_classification
# MAGIC
# MAGIC Esos atributos provienen de:
# MAGIC
# MAGIC `workspace.yelp_silver.slv_review_risk`
# MAGIC
# MAGIC Los resultados representan señales heurísticas y
# MAGIC candidatos a investigación.
# MAGIC
# MAGIC NO representan fraude confirmado.


# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOLD_SCHEMA = "yelp_gold"
GOV_SCHEMA = "yelp_gov"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"


# ------------------------------------------------------------
# Silver
# ------------------------------------------------------------

REVIEW_TABLE = (
    f"{SILVER}.slv_review"
)

USER_TABLE = (
    f"{SILVER}.slv_user"
)

BUSINESS_TABLE = (
    f"{SILVER}.slv_business"
)

EVALUATION_TABLE = (
    f"{SILVER}.slv_review_evaluation"
)

RISK_TABLE = (
    f"{SILVER}.slv_review_risk"
)


# ------------------------------------------------------------
# Governance
# ------------------------------------------------------------

RULE_TABLE = (
    f"{GOV}.evaluation_rule"
)

RULE_VERSION_TABLE = (
    f"{GOV}.evaluation_rule_version"
)

RUN_TABLE = (
    f"{GOV}.evaluation_run"
)


PIPELINE_NAME = (
    "06_gold_productos"
)

GOLD_MODEL_VERSION = "2.0.0"


print("=" * 72)
print("YELPCONNECT - GOLD TRUST & SAFETY")
print("=" * 72)

print(
    f"Pipeline : {PIPELINE_NAME}"
)

print(
    f"Version  : {GOLD_MODEL_VERSION}"
)

print("=" * 72)


# COMMAND ----------

# ============================================================
# 3. VALIDAR TABLAS REQUERIDAS
# ============================================================

required_tables = [

    REVIEW_TABLE,
    USER_TABLE,
    BUSINESS_TABLE,
    EVALUATION_TABLE,
    RISK_TABLE,
    RULE_TABLE,
    RULE_VERSION_TABLE,
    RUN_TABLE
]


for table_name in required_tables:

    if not spark.catalog.tableExists(
        table_name
    ):

        raise Exception(
            f"""
            ❌ Tabla requerida no encontrada:

            {table_name}
            """
        )


print(
    "✔ Tablas requeridas disponibles."
)


# COMMAND ----------

# ============================================================
# 4. CREAR SCHEMA GOLD
# ============================================================

spark.sql(
    f"""
    CREATE SCHEMA IF NOT EXISTS
    {CATALOG}.{GOLD_SCHEMA}
    """
)


print(
    f"✔ Schema disponible: {GOLD}"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 5. Seleccionar Evaluation Run publicable
# MAGIC
# MAGIC No basta con que el Run esté `SUCCESS`.
# MAGIC También debe existir Risk Consolidation para dicho Run.


# COMMAND ----------

# ============================================================
# 5. ÚLTIMO RUN SUCCESS CON RISK CONSOLIDATION
# ============================================================

risk_runs = (

    spark.table(
        RISK_TABLE
    )

    .select(
        "evaluation_run_id"
    )

    .distinct()
)


latest_run = (

    spark.table(
        RUN_TABLE
    )
    .alias("r")

    .filter(
        F.col(
            "r.execution_status"
        )
        ==
        "SUCCESS"
    )

    .join(
        risk_runs.alias("rr"),
        "evaluation_run_id",
        "inner"
    )

    .orderBy(
        F.desc(
            "r.started_at"
        )
    )

    .select(
        "evaluation_run_id",
        "pipeline_name",
        "pipeline_version",
        "dataset_version",
        "started_at",
        "finished_at"
    )

    .first()
)


if latest_run is None:

    raise Exception(
        """
        ❌ No existe un Evaluation Run SUCCESS
        que tenga Risk Consolidation.

        Ejecutar primero:

        03_trust_safety_evaluacion
        ↓
        05_risk_consolidation
        """
    )


EVALUATION_RUN_ID = (
    latest_run.evaluation_run_id
)


print(
    f"✔ Evaluation Run seleccionado: "
    f"{EVALUATION_RUN_ID}"
)


# COMMAND ----------

# ============================================================
# 6. VALIDAR CONSISTENCIA DEL RUN
# ============================================================

run_evaluations = (

    spark.table(
        EVALUATION_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )
)


run_risk = (

    spark.table(
        RISK_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )
)


evaluation_count = (
    run_evaluations.count()
)


evaluation_review_count = (

    run_evaluations

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


risk_count = (
    run_risk.count()
)


risk_distinct_reviews = (

    run_risk

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


print(
    f"Evaluaciones Rule-Level : "
    f"{evaluation_count:,}"
)

print(
    f"Reviews evaluadas       : "
    f"{evaluation_review_count:,}"
)

print(
    f"Reviews Risk            : "
    f"{risk_count:,}"
)


if risk_count == 0:

    raise Exception(
        """
        ❌ Risk Consolidation no tiene registros
        para el Evaluation Run seleccionado.
        """
    )


if (
    risk_count
    !=
    evaluation_review_count
):

    raise Exception(
        f"""
        ❌ Inconsistencia entre Evaluation y Risk.

        Reviews Evaluation:
        {evaluation_review_count}

        Reviews Risk:
        {risk_count}
        """
    )


if (
    risk_distinct_reviews
    !=
    risk_count
):

    raise Exception(
        """
        ❌ slv_review_risk no cumple:

        1 Review × 1 Evaluation Run
        """
    )


print(
    "✔ Evaluation Run consistente para publicación Gold."
)


# COMMAND ----------

# ============================================================
# HELPER
# ============================================================

def save_gold(
    df,
    table_name
):

    full_name = (
        f"{GOLD}.{table_name}"
    )

    (
        df

        .write

        .format("delta")

        .mode("overwrite")

        .option(
            "overwriteSchema",
            "true"
        )

        .saveAsTable(
            full_name
        )
    )

    print(
        f"✔ {full_name}"
    )


# COMMAND ----------

# MAGIC %md
# MAGIC # 7. Dimensión Business


# COMMAND ----------

# ============================================================
# 7. GLD DIM BUSINESS
# ============================================================

dim_business = (

    spark.table(
        BUSINESS_TABLE
    )

    .select(

        "business_id",

        F.col(
            "nombre"
        ).alias(
            "business_name"
        ),

        F.col(
            "ciudad"
        ).alias(
            "city"
        ),

        F.col(
            "estado"
        ).alias(
            "state"
        ),

        F.col(
            "estrellas"
        ).alias(
            "business_stars"
        ),

        F.col(
            "n_resenas"
        ).alias(
            "review_count_source"
        ),

        F.col(
            "esta_abierto"
        ).alias(
            "is_open"
        ),

        F.col(
            "categorias"
        ).alias(
            "categories"
        )
    )

    .withColumn(
        "business_sk",

        F.sha2(
            F.col(
                "business_id"
            ),
            256
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    dim_business,
    "gld_dim_business"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 8. Dimensión User
# MAGIC
# MAGIC Se construye desde los usuarios que aparecen en Reviews.
# MAGIC
# MAGIC Cuando el usuario no existe en `slv_user`,
# MAGIC se conserva igualmente la clave y se marca:
# MAGIC
# MAGIC `user_profile_available = false`.


# COMMAND ----------

# ============================================================
# 8. GLD DIM USER
# ============================================================

review_users = (

    spark.table(
        REVIEW_TABLE
    )

    .select(
        "user_id"
    )

    .where(
        F.col(
            "user_id"
        ).isNotNull()
    )

    .distinct()
)


user_profile = (

    spark.table(
        USER_TABLE
    )

    .select(

        "user_id",

        F.col(
            "nombre"
        ).alias(
            "user_name"
        ),

        "yelping_since",
        "review_count",
        "fans",
        "average_stars"
    )
)


dim_user = (

    review_users
    .alias("ru")

    .join(
        user_profile.alias("u"),
        "user_id",
        "left"
    )

    .withColumn(
        "user_profile_available",

        F.col(
            "u.yelping_since"
        ).isNotNull()
    )

    .withColumn(
        "user_sk",

        F.sha2(
            F.col(
                "user_id"
            ),
            256
        )
    )

    .select(

        "user_sk",
        "user_id",

        F.col(
            "u.user_name"
        ).alias(
            "user_name"
        ),

        F.col(
            "u.yelping_since"
        ).alias(
            "yelping_since"
        ),

        F.col(
            "u.review_count"
        ).alias(
            "review_count_source"
        ),

        F.col(
            "u.fans"
        ).alias(
            "fans"
        ),

        F.col(
            "u.average_stars"
        ).alias(
            "average_stars"
        ),

        "user_profile_available"
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    dim_user,
    "gld_dim_user"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 9. Dimensión Rule
# MAGIC
# MAGIC Grano:
# MAGIC
# MAGIC **1 fila = 1 versión de regla**


# COMMAND ----------

# ============================================================
# 9. GLD DIM RULE
# ============================================================

dim_rule = (

    spark.table(
        RULE_VERSION_TABLE
    )
    .alias("v")

    .join(

        spark.table(
            RULE_TABLE
        )
        .alias("r"),

        F.col(
            "v.rule_id"
        )
        ==
        F.col(
            "r.rule_id"
        ),

        "inner"
    )

    .select(

        F.col(
            "v.rule_version_id"
        ),

        F.col(
            "v.rule_id"
        ),

        F.col(
            "r.rule_name"
        ),

        F.col(
            "r.rule_type"
        ),

        F.col(
            "r.business_definition"
        ),

        F.col(
            "r.owner"
        ),

        F.col(
            "r.active_flag"
        ),

        F.col(
            "v.version"
        ),

        F.col(
            "v.description"
        ),

        F.col(
            "v.parameters_json"
        ),

        F.col(
            "v.threshold"
        ),

        F.col(
            "v.weight"
        ),

        F.col(
            "v.status"
        ),

        F.col(
            "v.valid_from"
        )
    )

    .withColumn(
        "rule_sk",

        F.sha2(
            F.col(
                "rule_version_id"
            ),
            256
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    dim_rule,
    "gld_dim_rule"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 10. Dimensión Evaluation Run


# COMMAND ----------

# ============================================================
# 10. GLD DIM EVALUATION RUN
# ============================================================

dim_run = (

    spark.table(
        RUN_TABLE
    )

    .withColumn(
        "evaluation_run_sk",

        F.sha2(
            F.col(
                "evaluation_run_id"
            ),
            256
        )
    )

    .withColumn(
        "is_current_gold_run",

        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    dim_run,
    "gld_dim_evaluation_run"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 11. Fact Review Evaluation
# MAGIC
# MAGIC Grano:
# MAGIC
# MAGIC **1 Review × 1 Rule Version × 1 Evaluation Run**
# MAGIC
# MAGIC Esta tabla contiene la explicación detallada
# MAGIC de cada regla.


# COMMAND ----------

# ============================================================
# 11. GLD FACT REVIEW EVALUATION
# ============================================================

review_keys = (

    spark.table(
        REVIEW_TABLE
    )

    .select(
        "review_id",
        "user_id",
        "business_id",
        "review_ts",
        "stars"
    )
)


fact_review_evaluation = (

    run_evaluations

    .join(
        review_keys,
        "review_id",
        "left"
    )

    .withColumn(
        "user_sk",

        F.sha2(
            F.col(
                "user_id"
            ),
            256
        )
    )

    .withColumn(
        "business_sk",

        F.sha2(
            F.col(
                "business_id"
            ),
            256
        )
    )

    .withColumn(
        "rule_sk",

        F.sha2(
            F.col(
                "rule_version_id"
            ),
            256
        )
    )

    .withColumn(
        "evaluation_run_sk",

        F.sha2(
            F.col(
                "evaluation_run_id"
            ),
            256
        )
    )

    .withColumn(
        "review_date",
        F.to_date(
            "review_ts"
        )
    )

    .withColumn(
        "evaluation_count",
        F.lit(1)
    )

    .select(

        "evaluation_id",

        "review_id",

        "user_sk",
        "business_sk",
        "rule_sk",
        "evaluation_run_sk",

        "evaluation_run_id",

        "rule_id",
        "rule_version_id",

        "evaluation_result",

        # Score aportado por ESTA regla.
        "risk_score",

        "reason",

        "stars",
        "review_ts",
        "review_date",

        "evaluated_at",

        "evaluation_count"
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    fact_review_evaluation,
    "gld_fact_review_evaluation"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 12. Producto Gold Review Risk
# MAGIC
# MAGIC Este producto NO recalcula riesgo.
# MAGIC
# MAGIC Hereda el resultado de `05_risk_consolidation`.


# COMMAND ----------

# ============================================================
# 12. REVIEW DETAIL
# ============================================================

review_detail = (

    spark.table(
        REVIEW_TABLE
    )

    .select(
        "review_id",
        "review_text"
    )
)


business_detail = (

    spark.table(
        BUSINESS_TABLE
    )

    .select(

        "business_id",

        F.col(
            "nombre"
        ).alias(
            "business_name"
        ),

        F.col(
            "ciudad"
        ).alias(
            "city"
        ),

        F.col(
            "estado"
        ).alias(
            "state"
        )
    )
)


# COMMAND ----------

# ============================================================
# 13. GLD REVIEW RISK
# ============================================================

gold_review_risk = (

    run_risk

    .join(
        review_detail,
        "review_id",
        "left"
    )

    .join(
        business_detail,
        "business_id",
        "left"
    )

    .withColumn(
        "user_sk",

        F.sha2(
            F.col(
                "user_id"
            ),
            256
        )
    )

    .withColumn(
        "business_sk",

        F.sha2(
            F.col(
                "business_id"
            ),
            256
        )
    )

    .withColumn(
        "evaluation_run_sk",

        F.sha2(
            F.col(
                "evaluation_run_id"
            ),
            256
        )
    )

    .withColumn(
        "review_date",

        F.to_date(
            "review_ts"
        )
    )

    # --------------------------------------------------------
    # Power BI friendly
    # --------------------------------------------------------

    .withColumn(
        "signal_rules_csv",

        F.concat_ws(
            ", ",
            F.col(
                "signal_rules"
            )
        )
    )

    .withColumn(
        "not_evaluable_rules_csv",

        F.concat_ws(
            ", ",
            F.col(
                "not_evaluable_rules"
            )
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    gold_review_risk,
    "gld_review_risk"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 14. Cola de investigación
# MAGIC
# MAGIC Solo contiene:
# MAGIC
# MAGIC `requires_investigation = true`
# MAGIC
# MAGIC Actualmente esperamos aproximadamente:
# MAGIC
# MAGIC - P1 = 3
# MAGIC - P2 = 299
# MAGIC - Total = 302
# MAGIC
# MAGIC según la ejecución validada.


# COMMAND ----------

# ============================================================
# 14. GLD INVESTIGATION QUEUE
# ============================================================

investigation_queue = (

    gold_review_risk

    .filter(
        F.col(
            "requires_investigation"
        )
        ==
        True
    )

    .select(

        "review_id",

        "user_id",
        "user_sk",

        "business_id",
        "business_sk",

        "business_name",
        "city",
        "state",

        "review_ts",
        "review_date",

        "stars",
        "review_text",

        "signal_count",
        "signal_rules",
        "signal_rules_csv",

        "weighted_risk_score",
        "signal_level",

        "risk_classification",
        "investigation_priority",

        "not_evaluable_count",
        "not_evaluable_rules",
        "evaluation_coverage_pct",
        "has_incomplete_evaluation",

        # ----------------------------------------------------
        # Evidencia explicable
        # ----------------------------------------------------

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",

        "user_review_count",

        "user_business_review_count",
        "user_business_concentration",

        "text_length",
        "text_repeat_count",

        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_spike_ratio",

        # ----------------------------------------------------
        # Trazabilidad
        # ----------------------------------------------------

        "evaluation_run_id",
        "evaluation_run_sk",

        "feature_execution_id",
        "risk_consolidation_id",
        "risk_model_version"
    )

    .withColumn(
        "investigation_status",

        F.lit(
            "PENDING_REVIEW"
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    investigation_queue,
    "gld_investigation_queue"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 15. KPI Ejecutivo
# MAGIC
# MAGIC Tabla de una fila ideal para las tarjetas principales
# MAGIC de Power BI.


# COMMAND ----------

# ============================================================
# 15. GLD KPI OVERVIEW
# ============================================================

kpi_overview = (

    gold_review_risk

    .agg(

        F.count("*")
        .alias(
            "reviews_evaluated"
        ),

        F.sum(

            F.when(
                F.col(
                    "signal_count"
                )
                >
                0,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_with_signal"
        ),

        F.sum(

            F.when(
                F.col(
                    "requires_investigation"
                ),
                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_for_investigation"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P1",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p1"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P2",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p2"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P3",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p3"
        ),

        F.sum(

            F.when(
                F.col(
                    "has_incomplete_evaluation"
                ),
                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_incomplete_evaluation"
        ),

        F.round(
            F.avg(
                "weighted_risk_score"
            ),
            4
        )
        .alias(
            "avg_weighted_risk_score"
        ),

        F.max(
            "weighted_risk_score"
        )
        .alias(
            "max_weighted_risk_score"
        )
    )

    .withColumn(
        "signal_rate_pct",

        F.round(

            F.col(
                "reviews_with_signal"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            4
        )
    )

    .withColumn(
        "investigation_rate_pct",

        F.round(

            F.col(
                "reviews_for_investigation"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            4
        )
    )

    .withColumn(
        "evaluation_run_id",

        F.lit(
            EVALUATION_RUN_ID
        )
    )

    .withColumn(
        "_gold_loaded_at",

        F.current_timestamp()
    )
)


save_gold(
    kpi_overview,
    "gld_kpi_trust_safety_overview"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 16. KPI por Business


# COMMAND ----------

# ============================================================
# 16. GLD KPI BUSINESS
# ============================================================

kpi_business = (

    gold_review_risk

    .groupBy(
        "business_id",
        "business_name",
        "city",
        "state"
    )

    .agg(

        F.count("*")
        .alias(
            "reviews_evaluated"
        ),

        F.sum(

            F.when(
                F.col(
                    "signal_count"
                )
                >
                0,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_with_signal"
        ),

        F.sum(

            F.when(
                F.col(
                    "requires_investigation"
                ),
                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_for_investigation"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P1",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p1"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P2",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p2"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P3",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p3"
        ),

        F.round(
            F.avg(
                "weighted_risk_score"
            ),
            4
        )
        .alias(
            "avg_weighted_risk_score"
        ),

        F.max(
            "weighted_risk_score"
        )
        .alias(
            "max_weighted_risk_score"
        )
    )

    .withColumn(
        "signal_rate_pct",

        F.round(

            F.col(
                "reviews_with_signal"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            2
        )
    )

    .withColumn(
        "investigation_rate_pct",

        F.round(

            F.col(
                "reviews_for_investigation"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            2
        )
    )

    .withColumn(
        "evaluation_run_id",

        F.lit(
            EVALUATION_RUN_ID
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    kpi_business,
    "gld_kpi_trust_safety_business"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 17. KPI diario


# COMMAND ----------

# ============================================================
# 17. GLD KPI DAILY
# ============================================================

kpi_daily = (

    gold_review_risk

    .groupBy(
        "review_date"
    )

    .agg(

        F.count("*")
        .alias(
            "reviews_evaluated"
        ),

        F.sum(

            F.when(
                F.col(
                    "signal_count"
                )
                >
                0,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_with_signal"
        ),

        F.sum(

            F.when(
                F.col(
                    "requires_investigation"
                ),
                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_for_investigation"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P1",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p1"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P2",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p2"
        ),

        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P3",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p3"
        ),

        F.round(
            F.avg(
                "weighted_risk_score"
            ),
            4
        )
        .alias(
            "avg_weighted_risk_score"
        )
    )

    .withColumn(
        "signal_rate_pct",

        F.round(

            F.col(
                "reviews_with_signal"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            2
        )
    )

    .withColumn(
        "investigation_rate_pct",

        F.round(

            F.col(
                "reviews_for_investigation"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            2
        )
    )

    .withColumn(
        "evaluation_run_id",

        F.lit(
            EVALUATION_RUN_ID
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    kpi_daily,
    "gld_kpi_trust_safety_daily"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 18. KPI por regla
# MAGIC
# MAGIC Permite analizar:
# MAGIC
# MAGIC - cantidad evaluada
# MAGIC - señales
# MAGIC - NOT_EVALUABLE
# MAGIC - tasa de activación
# MAGIC
# MAGIC R007 no aparecerá mientras permanezca DRAFT,
# MAGIC porque no forma parte del Evaluation Run.


# COMMAND ----------

# ============================================================
# 18. GLD KPI RULE
# ============================================================

rule_names = (

    spark.table(
        RULE_TABLE
    )

    .select(
        "rule_id",
        "rule_name",
        "rule_type"
    )
)


kpi_rule = (

    run_evaluations

    .groupBy(
        "rule_id",
        "rule_version_id"
    )

    .agg(

        F.count("*")
        .alias(
            "reviews_evaluated"
        ),

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "SIGNAL",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "signals"
        ),

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NO_SIGNAL",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "no_signals"
        ),

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NOT_EVALUABLE",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "not_evaluable"
        )
    )

    .withColumn(
        "evaluable_reviews",

        F.col(
            "reviews_evaluated"
        )

        -

        F.col(
            "not_evaluable"
        )
    )

    .withColumn(
        "signal_rate_pct",

        F.round(

            F.when(

                F.col(
                    "evaluable_reviews"
                )
                >
                0,

                F.col(
                    "signals"
                )

                /

                F.col(
                    "evaluable_reviews"
                )

                *
                100
            ),

            4
        )
    )

    .withColumn(
        "evaluation_coverage_pct",

        F.round(

            F.col(
                "evaluable_reviews"
            )

            /

            F.col(
                "reviews_evaluated"
            )

            *
            100,

            2
        )
    )

    .join(
        rule_names,
        "rule_id",
        "left"
    )

    .withColumn(
        "evaluation_run_id",

        F.lit(
            EVALUATION_RUN_ID
        )
    )

    .withColumn(
        "_gold_loaded_at",
        F.current_timestamp()
    )
)


save_gold(
    kpi_rule,
    "gld_kpi_trust_safety_rule"
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 19. Validación Gold


# COMMAND ----------

# ============================================================
# 19. TABLAS GOLD ESPERADAS
# ============================================================

gold_tables = [

    "gld_dim_business",
    "gld_dim_user",
    "gld_dim_rule",
    "gld_dim_evaluation_run",

    "gld_fact_review_evaluation",

    "gld_review_risk",
    "gld_investigation_queue",

    "gld_kpi_trust_safety_overview",
    "gld_kpi_trust_safety_business",
    "gld_kpi_trust_safety_daily",
    "gld_kpi_trust_safety_rule"
]


validation = []


for table_name in gold_tables:

    full_name = (
        f"{GOLD}.{table_name}"
    )

    count = (

        spark.table(
            full_name
        )

        .count()
    )

    validation.append(
        (
            table_name,
            count
        )
    )


validation_df = (

    spark.createDataFrame(
        validation,
        [
            "gold_product",
            "rows"
        ]
    )
)


display(
    validation_df
)


# COMMAND ----------

# ============================================================
# 20. VALIDACIONES CRÍTICAS
# ============================================================

gold_risk_count = (

    spark.table(
        f"{GOLD}.gld_review_risk"
    )

    .count()
)


gold_fact_count = (

    spark.table(
        f"{GOLD}.gld_fact_review_evaluation"
    )

    .count()
)


gold_queue_count = (

    spark.table(
        f"{GOLD}.gld_investigation_queue"
    )

    .count()
)


expected_queue_count = (

    run_risk

    .filter(
        F.col(
            "requires_investigation"
        )
        ==
        True
    )

    .count()
)


if (
    gold_risk_count
    !=
    risk_count
):

    raise Exception(
        """
        ❌ gld_review_risk no coincide
        con slv_review_risk.
        """
    )


if (
    gold_fact_count
    !=
    evaluation_count
):

    raise Exception(
        """
        ❌ gld_fact_review_evaluation
        no coincide con las evaluaciones del Run.
        """
    )


if (
    gold_queue_count
    !=
    expected_queue_count
):

    raise Exception(
        """
        ❌ gld_investigation_queue
        no coincide con requires_investigation.
        """
    )


print(
    "✔ Validaciones Gold superadas."
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 21. Distribución de prioridades


# COMMAND ----------

display(

    spark.table(
        f"{GOLD}.gld_review_risk"
    )

    .groupBy(
        "investigation_priority"
    )

    .count()

    .orderBy(

        F.when(
            F.col(
                "investigation_priority"
            )
            ==
            "P1",
            1
        )

        .when(
            F.col(
                "investigation_priority"
            )
            ==
            "P2",
            2
        )

        .when(
            F.col(
                "investigation_priority"
            )
            ==
            "P3",
            3
        )

        .otherwise(
            4
        )
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 22. Top candidatos para investigación


# COMMAND ----------

display(

    spark.table(
        f"{GOLD}.gld_investigation_queue"
    )

    .select(

        "review_id",
        "business_name",
        "stars",

        "signal_count",
        "signal_rules_csv",

        "weighted_risk_score",
        "signal_level",

        "investigation_priority",

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_24h",

        "text_repeat_count",
        "business_spike_ratio"
    )

    .orderBy(

        F.when(
            F.col(
                "investigation_priority"
            )
            ==
            "P1",
            1
        )

        .when(
            F.col(
                "investigation_priority"
            )
            ==
            "P2",
            2
        )

        .otherwise(
            3
        ),

        F.desc(
            "weighted_risk_score"
        ),

        F.desc(
            "signal_count"
        )
    )

    .limit(100)
)


# COMMAND ----------

# MAGIC %md
# MAGIC # 23. KPI ejecutivo


# COMMAND ----------

display(

    spark.table(
        f"{GOLD}.gld_kpi_trust_safety_overview"
    )
)


# COMMAND ----------

# ============================================================
# 24. RESUMEN FINAL
# ============================================================

overview = (

    spark.table(
        f"{GOLD}.gld_kpi_trust_safety_overview"
    )

    .first()
)


print()
print("=" * 72)
print("GOLD TRUST & SAFETY FINALIZADO")
print("=" * 72)

print(
    f"Evaluation Run          : "
    f"{EVALUATION_RUN_ID}"
)

print(
    f"Reviews publicadas      : "
    f"{gold_risk_count:,}"
)

print(
    f"Evaluaciones publicadas : "
    f"{gold_fact_count:,}"
)

print(
    f"Con alguna señal        : "
    f"{overview['reviews_with_signal']:,}"
)

print(
    f"Para investigación      : "
    f"{overview['reviews_for_investigation']:,}"
)

print(
    f"P1                      : "
    f"{overview['priority_p1']:,}"
)

print(
    f"P2                      : "
    f"{overview['priority_p2']:,}"
)

print(
    f"P3                      : "
    f"{overview['priority_p3']:,}"
)

print()
print(
    "Power BI debe consumir únicamente "
    "productos workspace.yelp_gold."
)

print("=" * 72)


# COMMAND ----------

# MAGIC %md
# MAGIC # Definition of Done — Gold
# MAGIC
# MAGIC - [ ] Se seleccionó un Evaluation Run SUCCESS con Risk Consolidation.
# MAGIC - [ ] Gold NO recalcula Risk Score.
# MAGIC - [ ] Gold NO recalcula Prioridad.
# MAGIC - [ ] Gold NO recalcula `requires_investigation`.
# MAGIC - [ ] `gld_dim_business` publicada.
# MAGIC - [ ] `gld_dim_user` cubre usuarios presentes en Reviews.
# MAGIC - [ ] `gld_dim_rule` publicada.
# MAGIC - [ ] `gld_dim_evaluation_run` publicada.
# MAGIC - [ ] `gld_fact_review_evaluation` mantiene grano Review × Rule × Run.
# MAGIC - [ ] `gld_review_risk` mantiene una fila por Review.
# MAGIC - [ ] `gld_investigation_queue` contiene únicamente candidatos.
# MAGIC - [ ] KPI Overview publicado.
# MAGIC - [ ] KPI Business publicado.
# MAGIC - [ ] KPI Daily publicado.
# MAGIC - [ ] KPI Rule publicado.
# MAGIC - [ ] Las reglas son explicables.
# MAGIC - [ ] Se mantiene `evaluation_run_id`.
# MAGIC - [ ] No se etiqueta fraude confirmado.